# Squirrel multiclass detection — Original YOLOv5 Medium (v2)

Complete workflow: audited dataset → immutable YOLO conversion → model/GPU verification → memory probe → training → validation/test reports → CPU/GPU benchmarks → model package.

**Task:** eleven-class bounding-box detection, with ten species plus `generic_squirrel`.
**Recipe:** pretrained medium model, 640 pixels, standard native YOLO sampling/augmentations, seed 42, AMP, **at most 200 epochs and early-stopping patience 30**.

Run `uv sync --locked --all-extras` before selecting the project `.venv` kernel. Run cells in order. The training cell starts the full experiment; preparation and probe cells do not. Generated files stay under `output_yolov5/runs/<run-id>/`.

Recover an interrupted experiment by setting `RESUME_RUN` below and keeping its configuration unchanged. Complete checkpoints remain frozen for validation/test. Source COCO images and splits are never modified.

YOLO26 reports both its accuracy-oriented and NMS-free heads from independent loads of the same checkpoint. Original YOLOv5 uses `yolov5m.pt`, not the anchor-free YOLOv5u variant.

In [1]:
# 1. Nonsecret configuration and independent run folder.
from pathlib import Path
import json
from IPython.display import display, HTML, IFrame
from csci_635_ml_project.baseline import (
    default_config, create_run, prepare_run, probe_batches, train_run,
    predict_run, benchmark_run, package_run,
)
from csci_635_ml_project.baseline_reports import dataset_report, evaluate_run

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
DATA = ROOT / "data/squirrel-v2-audited-coco"
CFG = default_config("yolov5")  # epochs=200, patience=30, imgsz=640, seed=42
RESUME_RUN = None  # ROOT / "output_yolov5/runs/<run-id>"
RUN = create_run(ROOT, CFG, resume_run=RESUME_RUN, data=DATA)
print("Run:", RUN)
display(CFG)

Run: /home/ss3813/github/ML/CSCI-635-ML-Project/output_yolov5/runs/20260929T154035Z-d7f2b6


{'model': 'yolov5',
 'epochs': 200,
 'patience': 30,
 'imgsz': 640,
 'seed': 42,
 'batch_candidates': [16, 8, 4],
 'workers': 4,
 'amp': True,
 'save_period': 10,
 'optimizer': 'SGD',
 'prediction_floor': 0.0001,
 'max_detections': 100,
 'matching_iou': 0.5,
 'nms_iou': 0.45,
 'heads': ['accuracy'],
 'ultralytics_version': '8.4.164',
 'yolov5_commit': '402e17ddf820996f51a191cbb798376e1144f069',
 'benchmark_images': 200,
 'benchmark_warmup': 20,
 'cpu_threads': 1}

## 2. Dataset verification, EDA, and backend setup

Require the passing audited v3 manifest and matching annotation hashes. Export the exact splits and class mapping to a fingerprinted YOLO dataset; every subsequent use verifies its image and label hashes. Native caches use the converted copy. Backend setup pins the model/source version, records pretrained weights and environment hashes, and executes a CUDA kernel.

In [2]:
DATA_YAML = prepare_run(RUN)
class_counts = dataset_report(RUN, DATA)
display(class_counts)
print("YOLO dataset:", DATA_YAML)
display(IFrame(src=str((RUN / "reports/report.html").relative_to(ROOT)), width="100%", height=700))

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/home/ss3813/github/ML/CSCI-635-ML-Project/output_yolov5/runs/20260929T154035Z-d7f2b6/.ultralytics/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.


,split,class_name,images,annotations
0,train,Callosciurus erythraeus,501,533
1,train,Sciurus aureogaster,534,570
2,train,Sciurus carolinensis,666,692
3,train,Sciurus granatensis,547,566
4,train,Sciurus griseus,581,655
5,train,Sciurus lis,409,449
6,train,Sciurus niger,551,606
7,train,Sciurus vulgaris,529,564
8,train,Tamiasciurus douglasii,574,591
9,train,Tamiasciurus hudsonicus,566,589


YOLO dataset: /home/ss3813/github/ML/CSCI-635-ML-Project/data/squirrel-v2-audited-yolo/96af70a0474fdebe/data.yaml


## 3. GPU memory probe

Try batches 16, 8, then 4 on a small seeded train/validation sample. Only CUDA OOM triggers a smaller-batch retry. Native nominal batch size is 64; resolved settings are saved. Probe weights are discarded for full training. Probe folders include batch-loss traces and a short Torch profiler trace where supported.

In [3]:
selected = probe_batches(RUN)
display(selected)

USDT:2026-09-29 11:40:56 2494402:2494402 SyncActivityProfilerHandler.cpp:39] profiler_start
train: weights=/home/ss3813/github/ML/CSCI-635-ML-Project/output_yolov5/pretrained/yolov5m.pt, cfg=, data=/home/ss3813/github/ML/CSCI-635-ML-Project/output_yolov5/runs/20260929T154035Z-d7f2b6/probe_data/data.yaml, hyp=/home/ss3813/github/ML/CSCI-635-ML-Project/output_yolov5/backend/402e17ddf820996f51a191cbb798376e1144f069/data/hyps/hyp.scratch-low.yaml, epochs=1, batch_size=16, imgsz=640, rect=False, resume=False, nosave=False, noval=False, noautoanchor=False, noplots=False, evolve=None, evolve_population=output_yolov5/backend/402e17ddf820996f51a191cbb798376e1144f069/data/hyps, resume_evolve=None, bucket=, cache=None, image_weights=False, device=0, multi_scale=False, single_cls=False, optimizer=SGD, sync_bn=False, workers=4, project=/home/ss3813/github/ML/CSCI-635-ML-Project/output_yolov5/runs/20260929T154035Z-d7f2b6/probes, name=batch-16-6d8690, exist_ok=True, quad=False, cos_lr=False, label_sm

{'batch_size': 16,
 'nominal_batch': 64,
 'attempts': [{'batch': 16, 'status': 'passed'}],
 'peak_vram_gib': 4.965756416320801}

## 4. Full training — this cell starts the experiment

Use the native optimizer/augmentation recipe with up to 200 epochs and 30-epoch early-stopping patience. YOLO26 selects its checkpoint by validation mAP50–95; original YOLOv5 uses 0.1 × AP50 + 0.9 × mAP50–95. Keep best/latest, periodic, and unstripped recovery checkpoints. Training logs, batch losses, epoch metrics, timing, VRAM, and loss curves stay in this run.

In [ ]:
if (RUN / "training_complete.json").exists():
    print("Training already completed; using the frozen best checkpoint.")
else:
    history = train_run(RUN)
    display(history.tail())

train: weights=/home/ss3813/github/ML/CSCI-635-ML-Project/output_yolov5/pretrained/yolov5m.pt, cfg=, data=/home/ss3813/github/ML/CSCI-635-ML-Project/data/squirrel-v2-audited-yolo/96af70a0474fdebe/data.yaml, hyp=/home/ss3813/github/ML/CSCI-635-ML-Project/output_yolov5/backend/402e17ddf820996f51a191cbb798376e1144f069/data/hyps/hyp.scratch-low.yaml, epochs=200, batch_size=16, imgsz=640, rect=False, resume=False, nosave=False, noval=False, noautoanchor=False, noplots=False, evolve=None, evolve_population=output_yolov5/backend/402e17ddf820996f51a191cbb798376e1144f069/data/hyps, resume_evolve=None, bucket=, cache=None, image_weights=False, device=0, multi_scale=False, single_cls=False, optimizer=SGD, sync_bn=False, workers=4, project=/home/ss3813/github/ML/CSCI-635-ML-Project/output_yolov5/runs/20260929T154035Z-d7f2b6, name=native, exist_ok=True, quad=False, cos_lr=False, label_smoothing=0.0, patience=30, freeze=[0], save_period=10, seed=42, local_rank=-1, entity=None, upload_dataset=False, 

## 5. Common validation and held-out test evaluation

Predict at confidence floor 0.0001 with maximum 100 detections. Select each head's threshold using validation micro F1 at matching IoU 0.50; ties choose the highest threshold. Freeze its checkpoint/hash and threshold before test scoring. Final AP/AR uses the same pycocotools evaluator as RF-DETR, rather than comparing framework-native AP implementations.

Reports include per-class AP/precision/recall/F1, aggregate metrics, PR/confidence curves, raw and normalized confusion matrices including background, and annotated successes/errors. Wrong species contributes one false positive and one false negative. Loss scales are model-specific.

In [ ]:
predict_run(RUN)
metrics = evaluate_run(RUN, DATA, heads=CFG["heads"])
for head, splits in metrics.items():
    print(head, {split: result["coco"]["mAP_50_95"] for split, result in splits.items()})
print("Reports:", RUN / "reports")

## 6. Local GPU and CPU latency

Benchmark identical seeded validation images, batch one, FP32, 20 warm-up calls, and one CPU thread. Include preprocessing/inference/postprocessing; exclude decoding. GPU timing synchronizes CUDA. No compilation or TF32. Save every image timing, mean/median/p95, FPS, hardware, checkpoint, threshold, and timing protocol. These measure local PyTorch performance; deployment-device timing requires a separate experiment.

In [ ]:
benchmarks = benchmark_run(RUN)
import pandas as pd
display(pd.DataFrame([{k: row[k] for k in ("head", "device", "mean_ms", "median_ms", "p95_ms", "throughput_fps")} for row in benchmarks]))

## 7. Package and optional explicit tracking

Export verified native best weights, class order, frozen thresholds, checkpoint/dataset hashes, model card, and report archive. Automatic framework tracking is disabled. If ClearML credentials exist privately in `.env`, this cell uploads explicit reports and metrics. Failures preserve local reports and a retry queue; no credentials are written into notebook outputs.

In [ ]:
best_weights = package_run(RUN)
print("Verified best weights:", best_weights)
print("Recovery checkpoint:", RUN / "native/weights/last_resumable.pt")
print("Report archive:", RUN / "reports.zip")

## References

- [YOLO26 models and inference heads](https://docs.ultralytics.com/models/yolo26/)
- [Original YOLOv5 source](https://github.com/ultralytics/yolov5)
- [YOLOv5 versus YOLOv5u compatibility](https://docs.ultralytics.com/models/yolov5/)
- [Ultralytics training settings](https://docs.ultralytics.com/modes/train/)